# IR + Raman + H/C → SMILES（Xinyu 协议）

使用 Xinyu 保存的四模态 checkpoint、H/C 参考与归一化；生成按原评测脚本去重和 ITM 重排。

本 notebook 展示已记录的真实评测结果。模型推理需显式开启，始终不进行训练。

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'vibrational.generation.hc'
example_input = {'ir': '1024 min-max-normalized intensities', 'raman': '1024 min-max-normalized intensities', 'wavenumber_cm1': [400, 4000], 'nmr_h_shifts_ppm': [1.2, 3.65], 'nmr_c_shifts_ppm': [18.2, 58.3], 'nmr_reference_id': 'must match checkpoint', 'formula': 'not supplied to Vib2Mol'}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "ir": "1024 min-max-normalized intensities",
  "raman": "1024 min-max-normalized intensities",
  "wavenumber_cm1": [
    400,
    4000
  ],
  "nmr_h_shifts_ppm": [
    1.2,
    3.65
  ],
  "nmr_c_shifts_ppm": [
    18.2,
    58.3
  ],
  "nmr_reference_id": "must match checkpoint",
  "formula": "not supplied to Vib2Mol"
}


## 已记录的评测

保留样本量、输入条件和来源文件；流程执行成功不等于精度达标。

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "vibrational.generation.hc",
    "dataset_name": null,
    "status": "passed",
    "metrics": {
      "exact_canonical_top1": 0.75,
      "exact_canonical_top3": 0.890625,
      "exact_canonical_top5": 0.890625,
      "exact_canonical_top10": 0.890625,
      "valid_smiles_fraction": 0.959375,
      "duplicate_valid_beams": 455,
      "examples": 64
    },
    "given_nmr": true,
    "beam_size": 10,
    "reranked_metrics": {
      "exact_canonical_top1": 0.6875,
      "exact_canonical_top3": 0.84375,
      "exact_canonical_top5": 0.890625,
      "exact_canonical_top10": 0.890625
    },
    "source_counts": {
      "ZINCV2": 28,
      "SDBS": 2,
      "PCQM4MV2": 11,
      "WILEYIR": 1,
      "ZINCV1": 7,
      "RXN": 5,
      "NISTIR": 1,
      "QM9": 9
    },
    "selection": {
      "read": 20390,
      "duplicate_structure": 3662,
      "eligible_distinct_structures": 16728,
      "selected": 64
    },
    "sampling": "reservoir",
    "seed": 42,
    "source_file": 

## 可选：冻结权重推理

设置 `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` 并配置工作区和模型环境。缺少权重时跳过或报错，不会启动训练。

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
